# 2.3 Dizilerde Hesaplama: Evrensel Fonksiyonlar

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/02-numpy/03-computation-ufuncs.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Computation on NumPy Arrays: Universal Functions

Şimdiye kadar NumPy'nin temel yapı taşlarını konuştuk. Önümüzdeki birkaç bölümde NumPy'nin Python veri bilimi dünyasında neden bu kadar önemli olduğunu ele alacağız: diziler üzerinde hesaplamayı kolay ve esnek biçimde optimize eden arayüz sunması.

NumPy dizilerinde hesaplama çok hızlı veya çok yavaş olabilir. Hızlı yapmanın anahtarı, genelde NumPy'nin evrensel fonksiyonları (ufunc) aracılığıyla uygulanan vektörize işlemler kullanmaktır. Bu bölüm NumPy ufunc'larına duyulan ihtiyacı açıklar; dizi elemanları üzerinde tekrarlanan hesaplamaları çok daha verimli kılarlar. Ardından NumPy paketindeki en yaygın ve yararlı aritmetik ufunc'lardan birçoğunu tanıtır.

## Döngülerin yavaşlığı

Python'un varsayılan uygulaması (CPython) bazı işlemleri çok yavaş yapar. Bunun bir nedeni dilin dinamik ve yorumlanmış doğasıdır; tipler esnektir, bu yüzden işlem dizileri C ve Fortran gibi dillerde olduğu gibi verimli makine koduna derlenemez. Son yıllarda bu zayıflığı gidermeye yönelik çeşitli girişimler oldu: [PyPy](http://pypy.org/) (JIT derlemeli Python), [Cython](http://cython.org) (Python'u derlenebilir C'ye dönüştürür) ve [Numba](http://numba.pydata.org/) (Python kod parçalarını hızlı LLVM bayt koduna çevirir). Her birinin güçlü ve zayıf yönleri vardır; ancak üçünün de standart CPython motorunun erişimini ve popülaritesini geçtiği söylenebilir.

Python'un göreli yavaşlığı genelde birçok küçük işlemin tekrarlandığı durumlarda ortaya çıkar; örneğin her eleman üzerinde işlem yapmak için dizi üzerinde döngü kurmak. Değerlerden oluşan bir dizimiz olduğunu ve her birinin tersini hesaplamak istediğimizi düşünelim. Doğrudan bir yaklaşım şöyle görünebilir:


In [ ]:
# ters_dongu.py
import numpy as np
rng = np.random.default_rng(seed=1701)

def compute_reciprocals(values):
    output = np.empty(len(values))
    for i in range(len(values)):
        output[i] = 1.0 / values[i]
    return output

values = rng.integers(1, 10, size=5)
print(compute_reciprocals(values))



Bu uygulama C veya Java geçmişi olan birine doğal gelebilir. Ancak büyük girdi için çalışma süresini ölçersek bu işlemin çok yavaş olduğunu — belki de şaşırtıcı derecede — görürüz. IPython'un %timeit sihirli komutuyla (1.7 Profil ve Kod Zamanlama) kıyaslayacağız:


In [ ]:
# timeit_dongu.py
big_array = rng.integers(1, 100, size=1000000)
# IPython'da: %timeit compute_reciprocals(big_array)
# Beklenen çıktı (kitap): ~2.6 s ± 192 ms



Bu milyon işlemi hesaplayıp sonucu saklamak birkaç saniye sürer! Darboğaz işlemlerin kendisi değil; CPython'un döngünün her turunda yaptığı tip denetimi ve fonksiyon yönlendirmesidir. Her ters alma işleminde Python önce nesnenin tipine bakar ve o tip için doğru fonksiyonu dinamik olarak arar. Derlenmiş kodda tip önceden bilinirdi ve sonuç çok daha verimli hesaplanırdı.

> **Not**
>

## Ufunc'lara giriş

Birçok işlem türü için NumPy, tam da bu tür statik tipli, derlenmiş rutinlere uygun bir arayüz sunar. Buna vektörize işlem denir. Buradaki eleman bazlı bölme gibi basit işlemlerde vektörizasyon, Python aritmetik operatörlerini doğrudan dizi üzerinde kullanmak kadar kolaydır. Döngü NumPy'nin altındaki derlenmiş katmana itilir; yürütme çok hızlanır.

İki yaklaşımın sonuçlarını karşılaştıralım:


In [ ]:
# vektor_karsilastirma.py
print(compute_reciprocals(values))
print(1.0 / values)



Büyük dizi için yürütme süresine bakarsak Python döngüsünden birkaç büyüklük mertebesi daha hızlı bittiğini görürüz:


In [ ]:
# IPython'da: %timeit (1.0 / big_array)
# Beklenen çıktı (kitap): ~2.54 ms ± 383 µs



NumPy'deki vektörize işlemler ufunc'lar aracılığıyla uygulanır; asıl amaçları NumPy dizilerindeki değerler üzerinde tekrarlanan işlemleri hızlı yürütmektir. Ufunc'lar son derece esnektir — skaler ile dizi arasında işlem gördük; iki dizi arasında da işlem yapabilirler:


In [ ]:
# ufunc_iki_dizi.py
print(np.arange(5) / np.arange(1, 6))



Ufunc işlemleri tek boyutla sınırlı değildir; çok boyutlu dizilerde de çalışırlar:


In [ ]:
# ufunc_cok_boyut.py
x = np.arange(9).reshape((3, 3))
print(2 ** x)



Ufunc ile vektörize hesaplamalar, Python döngüsüyle yazılan eşdeğerlerden neredeyse her zaman daha verimlidir — özellikle diziler büyüdükçe. NumPy betiğinde böyle bir döngü gördüğünüzde vektörize ifadeyle değiştirilip değiştirilemeyeceğini düşünün.

## NumPy ufunc'larını keşfetme

Ufunc'lar iki türde gelir: tek girdi üzerinde çalışan unary ufunc'lar ve iki girdi üzerinde çalışan binary ufunc'lar. Her iki türün örneklerini göreceğiz.

### Dizi aritmetiği

NumPy ufunc'ları Python'un yerleşik aritmetik operatörlerini kullandıkları için doğal hissettirir. Standart toplama, çıkarma, çarpma ve bölme kullanılabilir:


In [ ]:
# aritmetik_temel.py
x = np.arange(4)
print("x      =", x)
print("x + 5  =", x + 5)
print("x - 5  =", x - 5)
print("x * 2  =", x * 2)
print("x / 2  =", x / 2)
print("x // 2 =", x // 2)  # taban bölme



Negasyon için unary ufunc, üs alma için **, mod için % operatörü de vardır:


In [ ]:
# aritmetik_ek.py
print("-x     = ", -x)
print("x ** 2 = ", x ** 2)
print("x % 2  = ", x % 2)



Bunlar istediğiniz gibi zincirlenebilir; standart işlem önceliği korunur:


In [ ]:
# aritmetik_zincir.py
print(-(0.5*x + 1) ** 2)



Tüm bu aritmetik işlemler NumPy'ye gömülü belirli ufunc'ların kolay sarmalayıcılarıdır. Örneğin + operatörü add ufunc'ının sarmalayıcısıdır:


In [ ]:
# np_add.py
print(np.add(x, 2))



NumPy'de uygulanan aritmetik operatörlerin tablosu:

Ayrıca Boolean/bitwise operatörler vardır; bunları 2.6 Karşılaştırmalar, Maskeler ve Boolean Mantığı bölümünde ele alacağız.

### Mutlak değer

NumPy Python'un yerleşik aritmetik operatörlerini anladığı gibi yerleşik mutlak değer fonksiyonunu da anlar:


In [ ]:
# abs_builtin.py
x = np.array([-2, -1, 0, 1, 2])
print(abs(x))



Karşılık gelen NumPy ufunc'ı np.absolute'dır; np.abs takma adı da kullanılır:


In [ ]:
# np_abs.py
print(np.absolute(x))
print(np.abs(x))



Bu ufunc karmaşık sayıları da işler; bu durumda büyüklük (magnitude) döner:


In [ ]:
# abs_karmasik.py
x = np.array([3 - 4j, 4 - 3j, 2 + 0j, 0 + 1j])
print(np.abs(x))



### Trigonometrik fonksiyonlar

NumPy birçok yararlı ufunc sunar; veri bilimcisi için en kullanışlılarından biri trigonometrik fonksiyonlardır. Açılardan oluşan bir dizi tanımlayarak başlayalım:


In [ ]:
# trig_tanim.py
theta = np.linspace(0, np.pi, 3)



Bu değerler üzerinde trigonometrik fonksiyonları hesaplayabiliriz:


In [ ]:
# trig_hesap.py
print("theta      = ", theta)
print("sin(theta) = ", np.sin(theta))
print("cos(theta) = ", np.cos(theta))
print("tan(theta) = ", np.tan(theta))



Değerler makine hassasiyetiyle hesaplanır; bu yüzden sıfır olması gereken değerler her zaman tam sıfır olmayabilir. Ters trigonometrik fonksiyonlar da mevcuttur:


In [ ]:
# trig_ters.py
x = [-1, 0, 1]
print("x         = ", x)
print("arcsin(x) = ", np.arcsin(x))
print("arccos(x) = ", np.arccos(x))
print("arctan(x) = ", np.arctan(x))



### Üsler ve logaritmalar

NumPy ufunc'larında yaygın diğer işlemler üstel fonksiyonlardır:


In [ ]:
# exp_ornek.py
x = [1, 2, 3]
print("x   =", x)
print("e^x =", np.exp(x))
print("2^x =", np.exp2(x))
print("3^x =", np.power(3., x))



Üstellerin tersi olan logaritmalar da mevcuttur. Temel np.log doğal logaritmayı verir; 2 tabanlı veya 10 tabanlı logaritma için np.log2 ve np.log10 kullanılır:


In [ ]:
# log_ornek.py
x = [1, 2, 4, 10]
print("x        =", x)
print("ln(x)    =", np.log(x))
print("log2(x)  =", np.log2(x))
print("log10(x) =", np.log10(x))



Çok küçük girdilerde hassasiyeti korumak için özelleşmiş sürümler de vardır:


In [ ]:
# log_hassas.py
x = [0, 0.001, 0.01, 0.1]
print("exp(x) - 1 =", np.expm1(x))
print("log(1 + x) =", np.log1p(x))



x çok küçükken bu fonksiyonlar ham np.log veya np.exp kullanıldığında elde edilenden daha hassas değerler verir.

### Özelleşmiş ufunc'lar

NumPy'de hiperbolik trigonometri, bitwise aritmetik, karşılaştırma, radyan-derece dönüşümü, yuvarlama ve daha fazlası için birçok ufunc vardır. NumPy dokümantasyonuna bakmak ilginç işlevler ortaya çıkarır.

Daha özelleşmiş ufunc'lar için iyi bir kaynak scipy.special alt modülüdür. Veriniz üzerinde az bilinen bir matematiksel fonksiyon hesaplamak istiyorsanız büyük olasılıkla scipy.special'da vardır. İstatistik bağlamında karşınıza çıkabilecek birkaç örnek:


In [ ]:
# scipy_special.py
from scipy import special

# Gamma fonksiyonları (genelleştirilmiş faktöriyel) ve ilgili fonksiyonlar
x = [1, 5, 10]
print("gamma(x)     =", special.gamma(x))
print("ln|gamma(x)| =", special.gammaln(x))
print("beta(x, 2)   =", special.beta(x, 2))

# Hata fonksiyonu (Gauss integrali), tamamlayıcısı ve tersi
x = np.array([0, 0.3, 0.7, 1.0])
print("erf(x)  =", special.erf(x))
print("erfc(x) =", special.erfc(x))
print("erfinv(x) =", special.erfinv(x))



> **Not**
>

NumPy ve scipy.special'da çok daha fazla ufunc vardır. Paket dokümantasyonu çevrimiçi olduğundan "gamma function python" gibi bir arama genelde ilgili bilgiyi bulur.

## Gelişmiş ufunc özellikleri

Birçok NumPy kullanıcısı ufunc'ların tam özellik setini öğrenmeden kullanır. Burada birkaç özelleşmiş özelliği özetleyeceğim.

### Çıktı belirtme

Büyük hesaplamalarda sonucun yazılacağı diziyi belirtmek bazen yararlıdır. Tüm ufunc'larda bunu fonksiyonun out argümanıyla yapabilirsiniz:


In [ ]:
# ufunc_out.py
x = np.arange(5)
y = np.empty(5)
np.multiply(x, 10, out=y)
print(y)



Bu dizi görünümleriyle de kullanılabilir. Örneğin hesap sonucunu belirli bir dizinin her ikinci elemanına yazabiliriz:


In [ ]:
# ufunc_out_view.py
y = np.zeros(10)
np.power(2, x, out=y[::2])
print(y)



y[::2] = 2 ** x yazsaydık önce 2 ** x sonucunu tutan geçici bir dizi oluşur, ardından değerler y'ye kopyalanırdı. Küçük hesaplamada fark azdır; çok büyük dizilerde out argümanını dikkatli kullanmak bellek tasarrufu sağlar.

### Toplama (aggregation) işlemleri

İkili ufunc'lar için toplama işlemleri doğrudan nesne üzerinden hesaplanabilir. Örneğin bir diziyi belirli bir işlemle indirgemek (reduce) istiyorsak herhangi bir ufunc'ın reduce yöntemini kullanabiliriz. reduce, verilen işlemi dizi elemanlarına tekrar tekrar uygulayıncaya kadar tek sonuç kalana indirger.

Örneğin add ufunc'ında reduce çağrısı dizideki tüm elemanların toplamını verir:


In [ ]:
# reduce_add.py
x = np.arange(1, 6)
print(np.add.reduce(x))



Benzer şekilde multiply ufunc'ında reduce tüm elemanların çarpımını verir:


In [ ]:
# reduce_multiply.py
print(np.multiply.reduce(x))



Hesaplamanın tüm ara sonuçlarını saklamak istiyorsak accumulate kullanılabilir:


In [ ]:
# accumulate_ornek.py
print(np.add.accumulate(x))
print(np.multiply.accumulate(x))



Bu özel durumlar için sonuçları hesaplayan ayrı NumPy fonksiyonları vardır (np.sum, np.prod, np.cumsum, np.cumprod); bunları 2.4 Toplama İşlemleri bölümünde ele alacağız.

### Dış çarpım (outer product)

Son olarak herhangi bir ufunc, outer yöntemiyle iki farklı girdinin tüm çiftleri için çıktıyı hesaplayabilir. Tek satırda çarpım tablosu oluşturmak gibi işlemler yapmanızı sağlar:


In [ ]:
# outer_ornek.py
x = np.arange(1, 6)
print(np.multiply.outer(x, x))



ufunc.at ve ufunc.reduceat yöntemleri de yararlıdır; bunları 2.7 Gelişmiş İndeksleme bölümünde ele alacağız.

Ufunc'ların farklı şekil ve boyuttaki diziler arasında çalışabilmesi — broadcasting — o kadar önemlidir ki buna ayrı bir bölüm ayırıyoruz (2.5 Dizilerde Hesaplama: Broadcasting).

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      Döngü yerine vektörize bölme kullanın; np.add.reduce ve np.multiply.outer deneyin:
      
        import numpy as np
a = np.array([1, 2, 4, 8])
print(1.0 / a)
print("toplam:", np.add.reduce(a))
print("çarpım tablosu:\n", np.multiply.outer(a[:4], a[:4]))

## Ufunc'lar: Daha fazla bilgi

Evrensel fonksiyonlar hakkında daha fazla bilgi (mevcut fonksiyonların tam listesi dahil) NumPy ve SciPy dokümantasyon sitelerindedir.

Paketleri içe aktarıp IPython'da sekme tamamlama ve yardım (?) işlevini kullanarak bilgiye doğrudan erişebileceğinizi de unutmayın — 1.1 IPython'da Yardım ve Dokümantasyon.

> **Not**
>
